In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn as nn
from torchvision import models
from collections import Counter
import numpy as np
import os
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

In [2]:
print("CUDA disponible:", torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA disponible: True
12.9
NVIDIA GeForce RTX 3050


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  # luego mejoramos con padding si quieres
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
train_dataset = datasets.ImageFolder(
    root="../data/processed/train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root="../data/processed/val",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="../data/processed/test",
    transform=transform
)

In [7]:
print(train_dataset.classes)

['destroyed', 'major_damage', 'minor_damage', 'no_damage']


In [8]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,   # puedes subir si tienes CPU decente
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [9]:
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

In [10]:
for images, labels in val_loader:
    images = images.to(device)
    labels = labels.to(device)

In [11]:
for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)

In [12]:
model = models.resnet50(pretrained=True)

c:\Users\simon\Documents\Visual Studio Code\Imagenes-Pre-y-Post-Desastres\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\simon\Documents\Visual Studio Code\Imagenes-Pre-y-Post-Desastres\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [13]:
writer = SummaryWriter("runs/disaster_model")

In [14]:
num_classes = 4

model.fc = nn.Linear(model.fc.in_features, num_classes)

In [15]:
model = model.to(device)

In [16]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [17]:
# cuenta clases en train
counts = Counter()

for cls in os.listdir("../data/processed/train"):
    counts[cls] = len(os.listdir(f"../data/processed/train/{cls}"))

print(counts)

Counter({'no_damage': 95262, 'major_damage': 13854, 'minor_damage': 13599, 'destroyed': 8966})


In [18]:
total = sum(counts.values())

class_weights = []

for cls in train_dataset.classes:
    weight = total / counts[cls]
    class_weights.append(weight)

print(class_weights)


class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

[14.686705331251394, 9.504908329724268, 9.683138466063681, 1.3823035418110055]


In [19]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [20]:


optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [22]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    return total_loss / len(loader), acc

In [ ]:
from tqdm import tqdm

EPOCHS = 5
global_step = 0

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        #  log por batch
        writer.add_scalar("Loss/train_batch", loss.item(), global_step)
        global_step += 1

    train_loss /= len(train_loader)

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total

    #  log por epoch
    writer.add_scalar("Loss/train_epoch", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

100%|██████████| 2058/2058 [06:56<00:00,  4.95it/s]



Epoch 1
Train Loss: 1.0085
Val Loss: 0.9166, Val Acc: 0.6218


 23%|██▎       | 481/2058 [01:48<05:06,  5.14it/s]

In [ ]:
writer.close()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.classes)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
print(train_dataset.classes)
print(class_weights)